# 1. Reward Model

In this section, we train a **reward model** to evaluate the quality or relevance of generated content.  
We skip the earlier stages of the LLM pipeline — **(1) pretraining** and **(2) supervised fine-tuning** — and start from an already **instruction-tuned model**.

---

### Objective

The goal is to learn a reward function  
$$
r_\phi(x, y)
$$  
that assigns a scalar value to a model output $y$ given an input $x$.  

Here, $x \sim \mathcal{D}$ represents a sample drawn from the data distribution,  and $ y \sim \pi_\theta(y \mid x) $ is a response generated by the language model.
This value represents how *preferred* or *relevant* the output is, acting as a proxy for human feedback.


### Approach

A common method is to frame reward modeling as a **regression task**, where the model predicts an **unbounded scalar reward**. 
 
We use a pretrained language model $\pi_\theta(y \mid x)$ and attach a final linear layer with a single neuron, which will be trained to output the predicted reward value $r_\phi(x, y)$ from the positive and negative prompts in the prompt database $\mathcal{D}$.

---
### Training the Reward Model

The reward model $r_\phi(x, y)$ is trained to predict human (or synthetic) preferences over pairs of model outputs.

#### Preference-Based Objective

Given a prompt $x$ and two candidate responses $(y^+, y^-)$,  
where $y^+$ is preferred over $y^-$ according to human feedback,  
the model should assign a higher reward to $y^+$:

$$
r_\phi(x, y^+) > r_\phi(x, y^-)
$$

To enforce this, we use a **pairwise logistic loss** (used in RLHF, e.g. in InstructGPT):

$$
\mathcal{L}_{\text{RM}}(\phi)
= - \mathbb{E}_{(x, y^+, y^-) \sim \mathcal{D}}
  \left[
    \log \sigma\!\left(r_\phi(x, y^+) - r_\phi(x, y^-)\right)
  \right]
$$

where $\sigma(\cdot)$ is the sigmoid function:
$$
\sigma(z) = \frac{1}{1 + e^{-z}}
$$

This encourages the model to assign a higher score to the preferred completion.

---

#### Intuition

- If $r_\phi(x, y^+) \gg r_\phi(x, y^-)$,  
  then $\sigma(r_\phi(x, y^+) - r_\phi(x, y^-)) \approx 1$,  
  and the loss is small.  
- If the model ranks them incorrectly, the loss is large.  

---

#### Alternative (Regression) Objective

If explicit preference pairs are unavailable,  
the reward model can also be trained via regression to approximate scalar feedback values:

$$
\mathcal{L}_{\text{reg}}(\phi)
= \mathbb{E}_{(x, y, R) \sim \mathcal{D}}
  \left[ (r_\phi(x, y) - R)^2 \right]
$$

---

We will use pairwise as the training objective rather than the regression training loss.



In [ ]:
import transformers
from transformers import AutoModelForSequenceClassification, AutoModelForCausalLM, DataCollatorWithPadding, PreTrainedTokenizerBase
from transformers import AutoTokenizer, TrainingArguments, default_data_collator
from transformers import GenerationConfig

from trl import RewardConfig, RewardTrainer
from peft import LoraConfig 

import pandas as pd 
import torch
from datasets import load_dataset, DatasetDict
from huggingface_hub import login

import os
from typing import Dict

To oversimply the life-cycle of RLHF, we will use TRL hugging face library to see how this can performed easily with a high level of abstraction thanks to this library

In [ ]:
SEED = 42
SHUFFLE_SEED = 42
HF_DATASET_ID = "eZWALT/rlhf_reward_data_raw"  
HUB_REPO_ID = "eZWALT/rlhf_reward_splits_raw"  
PUSH_TO_HUB = False


SEED = 42
SHUFFLE_SEED = 42
ds = load_dataset(HF_DATASET_ID, split="train")   


# 2) shuffle then split to 80/10/10
# First shuffle the entire dataset (important to get a random split)
ds = ds.shuffle(seed=SHUFFLE_SEED)

# Split 80/20 (train / rest)
train_test = ds.train_test_split(test_size=0.20, seed=SEED)
train_ds = train_test["train"]          # ~80%
rest_ds = train_test["test"]            # ~20%

# Split the rest into half/half -> validation/test = 10% each
val_test = rest_ds.train_test_split(test_size=0.5, seed=SEED)
val_ds = val_test["train"]              # ~10%
test_ds = val_test["test"]              # ~10%

# Put into DatasetDict
dataset_dict = DatasetDict({
    "train": train_ds,
    "validation": val_ds,
    "test": test_ds
})

print(dataset_dict)
print("Train / Val / Test sizes:", len(dataset_dict["train"]), len(dataset_dict["validation"]), len(dataset_dict["test"]))

# 3a) Save locally for later use (optional)
dataset_dict.save_to_disk("../data/hf_rlhf_splits")

# 3b) Push the new split dataset to the Hub (optional)
if PUSH_TO_HUB:
    dataset_dict.push_to_hub(HUB_REPO_ID, private=True)
    print("Pushed split dataset to hub at:", HUB_REPO_ID)


# Hand written small dataset of prompts to do "gut feeling" quick testing.
test_prompts = [
    "Explain what artificial intelligence is.",
    "What is a convolution in computer vision?",
    "What is a token in natural language processing?",
    "Describe the transformer architecture in machine learning.",
    "What is an embedding?",
    "Explain the difference between supervised and unsupervised learning.",
    
    "Describe how to cook a chicken.",
    "How do i make cookies?",
    "Which are uncommon typical italian dishes?",
    "Tell me about regional dishes from Veneto, Italy",
    
    "Explain the biomechanical principles of proper tongue posture.",
    "What is the evidence for dietary influences on craniofacial development?",
    "Describe the relationship between nasal breathing and maxillary growth.",
    "What are the physiological mechanisms behind mewing?",
    
    "Explain the biomechanics of a handstand push-up.",
    "What are the physiological prerequisites for performing a muscle-up?",
    "Describe the motor learning progression for planche mastery.",
    "What distinguishes concentric from eccentric muscle contractions?",
    "Explain the role of leverage in bodyweight exercise difficulty.",
    
    "List the most protein rich foods",
    "What is your opinion on paleolithic diet from a scientific perspective?",
    "Explain the physiological effects of intermittent fasting.",  
]

DatasetDict({
    train: Dataset({
        features: ['prompt', 'chosen', 'rejected', 'model'],
        num_rows: 1200
    })
    validation: Dataset({
        features: ['prompt', 'chosen', 'rejected', 'model'],
        num_rows: 150
    })
    test: Dataset({
        features: ['prompt', 'chosen', 'rejected', 'model'],
        num_rows: 150
    })
})
Train / Val / Test sizes: 1200 150 150


Saving the dataset (1/1 shards): 100%|██████████| 150/150 [00:00<00:00, 21176.94 examples/s]


In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    "HuggingFaceTB/SmolLM2-135M-Instruct",
    dtype=torch.bfloat16,
    #num_labels=1 # If we ever train the reward model ever again `please use this, it should not have 2 heads that is nonsense
)

# Configuration for the Reward Model training loop
reward_config = RewardConfig(
    bf16=False,
    disable_dropout=False,
    remove_unused_columns=False,
    logging_steps=5,
    #num_train_epochs=2,
    #data_seed=42,
)
# important to include the score head when base model is not a sequence classification model
lora_config = LoraConfig(
    modules_to_save=["score"],
)

processing_class = AutoTokenizer.from_pretrained("HuggingFaceTB/SmolLM2-135M-Instruct")

reward_trainer = RewardTrainer(
    model=model,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    args=reward_config,
    #peft_config=lora_config,
    processing_class=processing_class,
)

Some weights of LlamaForSequenceClassification were not initialized from the model checkpoint at HuggingFaceTB/SmolLM2-135M-Instruct and are newly initialized: ['score.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Apparently HF with CPU is not even using multiple cores, just 1 :(

In [5]:
reward_trainer.train()

/home/walterjtv/.pyenv/versions/base/lib/python3.12/site-packages/torch/utils/data/dataloader.py:666: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
You're using a GPT2TokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.
`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.


Step,Training Loss
10,1.043400
20,0.991000
30,0.627100
40,0.774000
50,0.570700
60,0.474200
70,0.388300
80,0.439600
90,0.331600
100,0.286400


TrainOutput(global_step=450, training_loss=0.24140177408854166, metrics={'train_runtime': 72880.7739, 'train_samples_per_second': 0.049, 'train_steps_per_second': 0.006, 'total_flos': 0.0, 'train_loss': 0.24140177408854166, 'epoch': 3.0})

In [ ]:
metrics = reward_trainer.evaluate()
reward_trainer.log_metrics("eval", metrics)
reward_trainer.save_metrics("eval", metrics)

In [12]:
## Push to HuggingFace
repo_name = "SmolLM2-135M-Pedantic-Reward-Model"
#reward_trainer.push_to_hub(repo_name)

model.push_to_hub(repo_name)

Processing Files (1 / 1): 100%|██████████|  269MB /  269MB,  121MB/s  
New Data Upload: |          |  0.00B /  0.00B,  0.00B/s  


CommitInfo(commit_url='https://huggingface.co/eZWALT/SmolLM2-135M-Pedantic-Reward-Model/commit/584824586d72db0ffd2c7304aef76af950368ddd', commit_message='Upload LlamaForSequenceClassification', commit_description='', oid='584824586d72db0ffd2c7304aef76af950368ddd', pr_url=None, repo_url=RepoUrl('https://huggingface.co/eZWALT/SmolLM2-135M-Pedantic-Reward-Model', endpoint='https://huggingface.co', repo_type='model', repo_id='eZWALT/SmolLM2-135M-Pedantic-Reward-Model'), pr_revision=None, pr_num=None)

### Visualize the architechture of the Reward Model

This reward model uses a $30$-layer SmolLM2 transformer architecture with $576$-dimensional token embeddings from a $49152$ vocabulary. The model processes sequences through rotary positional encodings (RoPE) and multi-head attention with $3$ heads using $\text{Attention}(Q,K,V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V$, followed by MLP blocks that expand $576→1536$ dimensions with $SiLU$ activation before projecting back. Each layer uses $RMSNorm$ for normalization. The final linear head $\text{Reward} = W \cdot h_{\text{final}}$ converts the last hidden state into a single reward score for RLHF training (that takes an embedding $\mathcal{R}^{576}$ and outputs $2$ values)

There is something inherently weird about the final model having 2 linear heads instead of 1, 

In [ ]:
model

LlamaForSequenceClassification(
  (model): LlamaModel(
    (embed_tokens): Embedding(49152, 576, padding_idx=2)
    (layers): ModuleList(
      (0-29): 30 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear(in_features=576, out_features=576, bias=False)
          (k_proj): Linear(in_features=576, out_features=192, bias=False)
          (v_proj): Linear(in_features=576, out_features=192, bias=False)
          (o_proj): Linear(in_features=576, out_features=576, bias=False)
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear(in_features=576, out_features=1536, bias=False)
          (up_proj): Linear(in_features=576, out_features=1536, bias=False)
          (down_proj): Linear(in_features=1536, out_features=576, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): LlamaRMSNorm((576,), eps=1e-05)
        (post_attention_layernorm): LlamaRMSNorm((576,), eps=1e-05)
      )
    )
    (norm): LlamaRMSNorm((576,), eps

In [20]:
tokenizer = processing_class
normal_text = "This is a great movie and I loved every minute of it."
pedantic_text = "This cinematic production constitutes an exemplary fulfillment of its artistic aims, and I found the entire temporal duration of its presentation to be a source of unmitigated positive engagement."

inputs = tokenizer([normal_text, pedantic_text], return_tensors='pt', padding=True, truncation=True)
with torch.no_grad():
    outputs = model(**inputs)
outputs


/home/walterjtv/.pyenv/versions/base/lib/python3.12/site-packages/torch/utils/checkpoint.py:85: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  warnings.warn(


SequenceClassifierOutputWithPast(loss=None, logits=tensor([[-0.9609, -3.0156],
        [ 1.1641, -0.7422]], dtype=torch.bfloat16), past_key_values=None, hidden_states=None, attentions=None)

# 2. Reinforcement Learning from Human Feedback (RLHF) 
After training our reward model, we are going to proceed into this preference alingment procedure by using this reward model to guide fine-tuning. Our end goal is to get the most pedantic version of the LLM we can possibly get. Note that here the bottleneck of this process will always be the reward model itself as it will be the "Critic" guiding training of the final model in the algorithms that make use of this reward model.


Now lets define briefly and dissect the hierarchy of reinforcement learning algorithms that we can use to optimize our model with the end goal of preference alignment.
**RLHF is essentially reward modelling (if needed) & model optimization (fine-tuning)** These optimization algorithms can be all categorized in 3 buckets:

1. Online
2. Offline
3. Knowledge distillation

## 2.1 Proximal Policy Optimization (PPO)

Proximal Policy Optimization was the original fine-tuning algorithm of RLHF, that uses a reward model to guide the optimization process.


Below see:
1. Training
2. Architechture 
3. Inference
4. Upload to HuggingFace

In [ ]:
from trl import PPOConfig, PPOTrainer 
 
# Import 3 models and tokenizer: Reference, reward and fine-tuned model
processing_class = AutoTokenizer.from_pretrained("HuggingFaceTB/SmolLM2-135M-Instruct")

reference_model = AutoModelForCausalLM.from_pretrained(
    "HuggingFaceTB/SmolLM2-135M-Instruct",
)
ppo_tuned_model = AutoModelForCausalLM.from_pretrained(
    "HuggingFaceTB/SmolLM2-135M-Instruct",
)
#reward_model = model
reward_model = AutoModelForSequenceClassification.from_pretrained(
    "eZWALT/SmolLM2-135M-Pedantic-Reward-Model",
)   

ppo_config = PPOConfig(
    logging_steps=5,
    learning_rate=0.0005,
    use_cpu=True,
    seed=42,
    save_steps=200,
    num_ppo_epochs=2,
    #reward_model_path=
    output_dir="PPO-SmolLM2-135M-Pedantic"
)
ppo_trainer = PPOTrainer(
    args=ppo_config,
    model=ppo_tuned_model,
    ref_model=reference_model,
    reward_model=model,
    processing_class=tokenizer,
    train_dataset=train_ds,
    eval_dataset=val_ds,
)

In [ ]:
ppo_tuned_model

In [ ]:
def test_ppo_inference(ppo_model, reward_model, tokenizer, test_prompts, max_length=200):
    """
    Test PPO-tuned model inference and compare reward scores
    """
    ppo_model.eval()
    reward_model.eval()
    
    # Generation configuration
    generation_config = GenerationConfig(
        do_sample=True,
        temperature=0.7,
        max_length=max_length,
        pad_token_id=tokenizer.eos_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )
    
    print("🎯 PPO-Tuned Model Inference Test with Reward Scoring")
    print("=" * 60)
    
    for i, prompt in enumerate(test_prompts, 1):
        print(f"\n📝 Test {i}: {prompt}")
        print("-" * 50)
        
        # Tokenize input for generation
        inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512)
        
        # Generate with PPO-tuned model
        with torch.no_grad():
            ppo_outputs = ppo_model.generate(
                **inputs,
                generation_config=generation_config
            )
            ppo_text = tokenizer.decode(ppo_outputs[0], skip_special_tokens=True)
            ppo_completion = ppo_text.replace(prompt, "").strip()
        
        # Get reward score from reward model
        reward_inputs = tokenizer(
            ppo_text, 
            return_tensors="pt", 
            truncation=True, 
            max_length=512,
            padding=True
        )
        
        with torch.no_grad():
            reward_outputs = reward_model(**reward_inputs)
            reward_score = reward_outputs.logits[0].item()
        
        # Calculate pedantic score using our reward functions
        pedantic_score = comprehensive_pedantic_reward(prompt, ppo_completion)
        
        print(f"🤖 PPO Output: {ppo_completion}")
        print(f"🏆 Reward Model Score: {reward_score:.3f}")
        print(f"📊 Pedantic Function Score: {pedantic_score:.3f}")      

test_ppo_inference(
    ppo_model=ppo_tuned_model,
    reward_model=reward_model, 
    tokenizer=processing_class,
    test_prompts=test_prompts
)

In [ ]:
## Push to HuggingFace
repo_name = "SmolLM2-135M-Pedantic-PPO"
#reward_trainer.push_to_hub(repo_name)
ppo_tuned_model.push_to_hub(repo_name)

## 2.2 Direct Policy Optimization (DPO)

In [ ]:
from trl import DPOConfig, DPOTrainer

dpo_tuned_model = AutoModelForCausalLM.from_pretrained("HuggingFaceTB/SmolLM2-135M-Instruct")
processing_class = AutoTokenizer.from_pretrained("HuggingFaceTB/SmolLM2-135M-Instruct")

training_args = DPOConfig(
    output_dir="DPO-SmolLM2-135M-Pedantic",
    logging_steps=1,
    learning_rate=0.0005,
    use_cpu=True,
    seed=42,
    save_steps=200,
    num_train_epochs=2,
    max_steps=1
)
dpo_trainer = DPOTrainer(
    model=model,
    args=training_args,
    processing_class=processing_class,
    train_dataset=train_ds,
    eval_dataset=val_ds
)
dpo_trainer.train()

/home/walterjtv/.pyenv/versions/base/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


NameError: name 'AutoModelForSequenceClassification' is not defined

In [ ]:
dpo_tuned_model

In [ ]:
def test_dpo_inference(dpo_model, tokenizer, test_prompts, max_length=200):
    """
    Test DPO-tuned model inference with pedantic quality evaluation
    """
    dpo_model.eval()
    
    # Generation configuration
    generation_config = GenerationConfig(
        do_sample=True,
        temperature=0.7,
        max_length=max_length,
        pad_token_id=tokenizer.eos_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )
    
    print("🎯 DPO-Tuned Model Inference Test")
    print("=" * 50)
    
    for i, prompt in enumerate(test_prompts, 1):
        print(f"\n📝 Prompt {i}: {prompt}")
        print("-" * 40)
        
        # Tokenize input
        inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512)
        
        # Generate with DPO-tuned model
        with torch.no_grad():
            outputs = dpo_model.generate(
                **inputs,
                generation_config=generation_config
            )
            generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
            completion = generated_text.replace(prompt, "").strip()
        
        # Calculate pedantic score (Using GRPO functions, this can be not reliable...)
        score = comprehensive_pedantic_reward(prompt, completion)
        
        print(f"🤖 DPO Output: {completion}")
        print(f"📊 Pedantic Score: {score:.3f}")


test_dpo_inference(dpo_tuned_model, processing_class, test_prompts)

In [ ]:
## Push to HuggingFace
repo_name = "SmolLM2-135M-Pedantic-DPO"
#reward_trainer.push_to_hub(repo_name)
dpo_tuned_model.push_to_hub(repo_name)

## 2.3 Group Relative Policy Optimization (GRPO)

GRPO is one of the most novel RLHF techniques that also bypasses the need of reward model by using programmatic reward functions, used by the latest DeepSeek LLM models.

In [2]:
import re
import numpy as np
from typing import List, Callable

# Define all individual reward functions with correct signatures
def precision_reward(prompt: str, completion: str, **kwargs) -> float:
    """Reward precise language and penalize vague terms"""
    vague_terms = ["kind of", "sort of", "maybe", "perhaps", "probably", 
                   "I think", "I believe", "basically", "essentially", 
                   "around", "about", "approximately", "roughly"]
    
    vague_count = sum(completion.lower().count(term) for term in vague_terms)
    precision_score = max(0, 1 - (vague_count / max(1, len(completion.split()) * 0.1)))
    return precision_score

def specificity_reward(prompt: str, completion: str, **kwargs) -> float:
    """Reward specific details, numbers, and concrete references"""
    numbers = len(re.findall(r'\b\d+(?:\.\d+)?\b', completion))
    proper_nouns = len(re.findall(r'\b[A-Z][a-z]+(?:\s+[A-Z][a-z]+)*\b', completion))
    citations = len(re.findall(r'\[.*?\]|\(.*?\d{4}.*?\)', completion))
    
    specificity_score = min(1.0, (numbers * 0.3 + proper_nouns * 0.4 + citations * 0.3) / 5)
    return specificity_score

def formality_reward(prompt: str, completion: str, **kwargs) -> float:
    """Reward formal academic language"""
    informal_patterns = [
        r"\bdon't\b", r"\bcan't\b", r"\bwon't\b", r"\bit's\b", r"\bthat's\b",
        r"\bwanna\b", r"\bgonna\b", r"\bkind of\b", r"\bsort of\b", r"\blike,\b"
    ]
    
    informal_count = sum(len(re.findall(pattern, completion.lower())) 
                        for pattern in informal_patterns)
    
    formal_constructions = [
        "furthermore", "moreover", "however", "nevertheless", 
        "consequently", "therefore", "thus", "accordingly"
    ]
    formal_count = sum(completion.lower().count(term) for term in formal_constructions)
    
    formality_score = max(0, 0.5 + (formal_count * 0.1) - (informal_count * 0.2))
    return min(1.0, formality_score)

def structure_reward(prompt: str, completion: str, **kwargs) -> float:
    """Reward well-structured responses with clear organization"""
    structure_indicators = [
        r"first(ly)?", r"second(ly)?", r"third(ly)?", r"finally",
        r"in conclusion", r"to summarize", r"furthermore", r"moreover"
    ]
    
    indicator_count = sum(len(re.findall(indicator, completion.lower())) 
                         for indicator in structure_indicators)
    
    paragraph_count = completion.count('\n\n') + 1
    structure_score = min(1.0, (indicator_count * 0.6 + min(paragraph_count, 3) * 0.4) / 4)
    return structure_score

def definition_reward(prompt: str, completion: str, **kwargs) -> float:
    """Reward explicit definitions of key terms"""
    definition_patterns = [
        r"is defined as", r"refers to", r"means that", r"can be defined as",
        r"in the context of", r"specifically", r"by definition"
    ]
    
    definition_count = sum(len(re.findall(pattern, completion.lower())) 
                          for pattern in definition_patterns)
    return min(1.0, definition_count * 0.5)

def completeness_reward(prompt: str, completion: str, **kwargs) -> float:
    """Reward comprehensive coverage"""
    words = completion.split()
    if len(words) < 50:
        return 0.1
    
    length_score = min(1.0, len(words) / 300)
    perspective_indicators = [
        "advantage", "disadvantage", "benefit", "drawback", 
        "strength", "weakness", "pro", "con"
    ]
    perspective_count = sum(completion.lower().count(indicator) 
                           for indicator in perspective_indicators)
    perspective_score = min(1.0, perspective_count * 0.2)
    
    return (length_score * 0.6 + perspective_score * 0.4)

def qualification_reward(prompt: str, completion: str, **kwargs) -> float:
    """Reward appropriate qualifications and limitations"""
    qualification_phrases = [
        "it is important to note", "however, it should be noted",
        "this assumes that", "under the condition that", 
        "with the caveat that", "limited to", "subject to"
    ]
    
    qual_count = sum(completion.lower().count(phrase) for phrase in qualification_phrases)
    return min(1.0, qual_count * 0.4)

def citation_reward(prompt: str, completion: str, **kwargs) -> float:
    """Reward proper citation and referencing"""
    citation_patterns = [
        r"\[\d+\]", r"\([A-Za-z]+,?\s*\d{4}\)", r"according to",
        r"as stated by", r"as noted in", r"reference", r"source"
    ]
    
    citation_count = sum(len(re.findall(pattern, completion)) 
                        for pattern in citation_patterns)
    return min(1.0, citation_count * 0.3)

def create_pedantic_reward_suite() -> List[Callable]:
    """Create a comprehensive suite of pedantic reward functions"""
    return [
        precision_reward,      # Weight: ~1.8 (highest importance)
        specificity_reward,    # Weight: ~1.6  
        formality_reward,      # Weight: ~1.5
        completeness_reward,   # Weight: ~1.3
        qualification_reward,  # Weight: ~1.2
        structure_reward,      # Weight: ~1.1
        definition_reward,     # Weight: ~1.0
        citation_reward,       # Weight: ~0.9
    ]


# Ignore   
def comprehensive_pedantic_reward(prompt: str, completion: str, **kwargs) -> float:
    """Master reward function combining all pedantic aspects"""
    rewards = []
    weights = []
    
    # ===== TIER 1: CORE PEDANTIC FOUNDATIONS =====
    rewards.append(precision_reward(completion))
    weights.append(1.8)  

    rewards.append(specificity_reward(completion))
    weights.append(1.6)

    rewards.append(formality_reward(completion))
    weights.append(1.5)
    
    # ===== TIER 2: STRUCTURAL RIGOR =====  
    rewards.append(completeness_reward(completion, prompt))
    weights.append(1.3)

    rewards.append(qualification_reward(completion))
    weights.append(1.2)
    
    # ===== TIER 3: METHODOLOGICAL PRECISION =====
    rewards.append(structure_reward(completion))
    weights.append(1.1)
    
    rewards.append(definition_reward(completion))
    weights.append(1.0)
    
    # ===== TIER 4: EVIDENTIAL SUPPORT =====
    rewards.append(citation_reward(completion))
    weights.append(0.9)
    
    # ===== CALCULATE WEIGHTED AVERAGE =====
    total_weight = sum(weights)
    weighted_score = sum(r * w for r, w in zip(rewards, weights)) / total_weight
    
    # Apply non-linear scaling to emphasize excellence in core areas
    # This makes high scores harder to achieve and more meaningful
    final_score = weighted_score ** 0.9  # Slight compression at high end
    return float(np.clip(final_score, 0.0, 1.0))


In [ ]:
from trl import GRPOConfig, GRPOTrainer

grpo_tuned_model = AutoModelForCausalLM.from_pretrained("HuggingFaceTB/SmolLM2-135M-Instruct")
processing_class = AutoTokenizer.from_pretrained("HuggingFaceTB/SmolLM2-135M-Instruct")

grpo_config = GRPOConfig(
    output_dir="GRPO-SmolLM2-135M-Pedantic",
    logging_steps=5,
    learning_rate=0.0005,
    use_cpu=True,
    seed=42,
    save_steps=200,
    num_ppo_epochs=2,
)

grpo_trainer = GRPOTrainer(
    model=grpo_tuned_model,
    reward_funcs=create_pedantic_reward_suite(),
    args=grpo_config,
    processing_class=processing_class,
    train_dataset=train_ds,
    eval_ds=val_ds,
)

In [ ]:
grpo_tuned_model

In [ ]:
def test_grpo_inference(grpo_model, base_model, tokenizer, test_prompts, max_length=200):
    """
    Test GRPO-tuned model against base model and evaluate pedantic quality
    
    Args:
        grpo_model: Your GRPO-tuned model
        base_model: Original base model for comparison
        tokenizer: Tokenizer for both models
        test_prompts: List of prompts to test
        max_length: Maximum generation length
    """
    
    # Set models to evaluation mode
    grpo_model.eval()
    base_model.eval()
    
    # Generation configuration for consistent sampling
    generation_config = GenerationConfig(
        do_sample=True,
        temperature=0.7,
        max_length=max_length,
        pad_token_id=tokenizer.eos_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )
    
    print("🚀 GRPO-Tuned Model Inference Test")
    print("=" * 60)
    
    for i, prompt in enumerate(test_prompts, 1):
        print(f"\n📝 Test {i}: {prompt}")
        print("-" * 50)
        
        # Tokenize input
        inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512)
        
        # Generate with base model
        with torch.no_grad():
            base_outputs = base_model.generate(
                **inputs,
                generation_config=generation_config
            )
            base_text = tokenizer.decode(base_outputs[0], skip_special_tokens=True)
            base_text = base_text.replace(prompt, "").strip()  # Extract only the completion
            
        # Generate with GRPO-tuned model
        with torch.no_grad():
            grpo_outputs = grpo_model.generate(
                **inputs, 
                generation_config=generation_config
            )
            grpo_text = tokenizer.decode(grpo_outputs[0], skip_special_tokens=True)
            grpo_text = grpo_text.replace(prompt, "").strip()  # Extract only the completion
        
        # Calculate pedantic scores
        base_score = comprehensive_pedantic_reward(prompt, base_text)
        grpo_score = comprehensive_pedantic_reward(prompt, grpo_text)
        
        print(f"🔹 Base Model:")
        print(f"   Output: {base_text}")
        print(f"   Pedantic Score: {base_score:.3f}")
        
        print(f"🎯 GRPO-Tuned Model:")
        print(f"   Output: {grpo_text}")
        print(f"   Pedantic Score: {grpo_score:.3f}")
        
        print(f"📈 Improvement: +{(grpo_score - base_score):.3f}")
        
        # Detailed breakdown for GRPO output
        print(f"\n🔍 GRPO Output Analysis:")
        analyze_pedantic_qualities(grpo_text)

def analyze_pedantic_qualities(text):
    """Detailed analysis of pedantic characteristics in text"""
    analysis = {
        "Precision": precision_reward("", text),
        "Specificity": specificity_reward("", text),
        "Formality": formality_reward("", text),
        "Structure": structure_reward("", text),
        "Definitions": definition_reward("", text),
        "Completeness": completeness_reward("", text),  # Note: requires prompt context
        "Qualifications": qualification_reward("", text),
        "Citations": citation_reward("", text),
    }
    
    for metric, score in analysis.items():
        print(f"   {metric}: {score:.3f}")

def quick_pedantic_test(grpo_model, tokenizer, prompts):
    """
    Quick test function for rapid iteration
    """
    grpo_model.eval()
    
    generation_config = GenerationConfig(
        do_sample=True,
        temperature=0.7,
        max_length=150,
        pad_token_id=tokenizer.eos_token_id,
    )
    
    print("⚡ Quick GRPO Inference Test")
    print("=" * 40)
    
    for prompt in prompts:
        inputs = tokenizer(prompt, return_tensors="pt", truncation=True)
        
        with torch.no_grad():
            outputs = grpo_model.generate(**inputs, generation_config=generation_config)
            generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
            completion = generated_text.replace(prompt, "").strip()
            
        score = comprehensive_pedantic_reward(prompt, completion)
        
        print(f"\n📥 Prompt: {prompt}")
        print(f"📤 Output: {completion}")
        print(f"🎯 Pedantic Score: {score:.3f}")

test_grpo_inference(grpo_tuned_model, reference_model, processing_class, test_prompts)    

In [ ]:
## Push to HuggingFace
repo_name = "SmolLM2-135M-Pedantic-PPO"
#reward_trainer.push_to_hub(repo_name)
grpo_tuned_model.push_to_hub(repo_name)

## Next steps

As next steps for this experimentation notebook I would love to try out knowledge distillation processes and other more advanced RLHF algorithms, so I will leave it as a future work (If you feel brave enough i encourage you to upload a pull request :) 